In [ ]:
import os

os.environ["TRANSFORMERS_VERBOSITY"] = "info"

from dataclasses import dataclass
from typing import Callable, cast

import torch
import torch.nn.functional as F
from torch.optim import AdamW

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
@dataclass
class GRPOConfig:
    G: int = 16  # group size — completions sampled per prompt
    beta: float = 0.01  # KL penalty coefficient
    epsilon: float = 0.2  # PPO-style clip range (optional)
    lr: float = 1e-5
    max_new_tokens: int = 128
    temperature: float = 1


SYSTEM_PROMPT = "You are a expert mathematician who answers questions accurately and concisely. Answer each question step by step, showing your work."


def make_messages(question: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]


# ── Core GRPO Loss ────────────────────────────────────────────────────────────


def compute_grpo_loss(
    policy_logps: torch.Tensor,  # (G,) — log probs of each completion under current policy
    ref_logps: torch.Tensor,  # (G,) — log probs under frozen reference model
    rewards: torch.Tensor,  # (G,) — scalar reward for each completion
    beta: float = 0.01,
    epsilon: float = 0.2,
    old_logps: torch.Tensor | None = None,  # (G,) — for PPO-style clipping
) -> tuple[torch.Tensor, dict]:
    """
    GRPO loss for a single group of G completions sharing one prompt.

    1. Normalize rewards → advantages
    2. (Optionally) clip importance ratio like PPO
    3. Subtract KL(policy || ref) penalty
    """
    # 1. Group-relative advantage normalization
    mean_r = rewards.mean()
    std_r = rewards.std().clamp(min=1e-8)
    advantages = (rewards - mean_r) / std_r  # (G,)
    advantages = advantages.clamp(-5, 5)

    # 2. Importance ratio  π_θ / π_θ_old
    log_ratio = policy_logps - (old_logps if old_logps is not None else policy_logps.detach())
    log_ratio = log_ratio.clamp(-10, 10)
    ratio = log_ratio.exp()  # (G,)

    if old_logps is not None:
        clipped = ratio.clamp(1 - epsilon, 1 + epsilon)
        policy_loss = -torch.min(ratio * advantages, clipped * advantages).mean()
    else:
        policy_loss = -(ratio * advantages).mean()

    # 3. KL penalty: KL(policy || ref) ≈ log(π_θ / π_ref)
    kl = (policy_logps - ref_logps).mean()
    kl = kl.clamp(-10, 10)

    loss = policy_loss + beta * kl
    return loss, {
        "policy_loss": policy_loss.item(),
        "kl": kl.item(),
        "mean_reward": mean_r.item(),
        "advantages": advantages.tolist(),
    }


# ── Batched sequence log-prob helper ─────────────────────────────────────────


def batch_completion_logprobs(
    model,
    prompt_ids: torch.Tensor,       # (1, prompt_len)
    comp_ids_list: list[torch.Tensor],  # list of G tensors, each (1, comp_len_i)
    with_grad: bool = True,
) -> torch.Tensor:
    """
    Single batched forward pass for all G completions.
    Returns (G,) mean per-token log-probs, one per completion.
    Replaces G separate forward passes with 1, giving ~G× speedup.
    """
    G = len(comp_ids_list)
    prompt_len = prompt_ids.shape[1]
    pad_id = model.config.eos_token_id if model.config.eos_token_id is not None else 0

    # Pad all completions to the same length
    max_comp_len = max(c.shape[1] for c in comp_ids_list)
    padded = torch.full((G, max_comp_len), pad_id, dtype=torch.long, device=model.device)
    comp_mask = torch.zeros(G, max_comp_len, dtype=torch.bool, device=model.device)
    for i, cids in enumerate(comp_ids_list):
        L = cids.shape[1]
        if L > 0:
            padded[i, :L] = cids[0]
            comp_mask[i, :L] = True

    # Build full_ids: (G, prompt_len + max_comp_len)
    full_ids = torch.cat([prompt_ids.expand(G, -1), padded], dim=1)
    attn_mask = torch.cat(
        [torch.ones(G, prompt_len, dtype=torch.long, device=model.device), comp_mask.long()],
        dim=1,
    )

    ctx = torch.enable_grad() if with_grad else torch.no_grad()
    with ctx:
        logits = model(full_ids, attention_mask=attn_mask).logits  # (G, T, vocab)

    # Shift: logits[t] predicts token[t+1]
    shift_logits = logits[:, :-1]          # (G, T-1, vocab)
    shift_labels = full_ids[:, 1:]        # (G, T-1)

    lp = F.log_softmax(shift_logits, dim=-1)

    # Completion tokens in shifted view start at prompt_len - 1
    comp_start = prompt_len - 1
    comp_lp = lp[:, comp_start:]          # (G, max_comp_len, vocab)
    comp_labels = shift_labels[:, comp_start:]  # (G, max_comp_len)

    # Gather per-token log-probs and mask padding
    token_lps = comp_lp.gather(-1, comp_labels.unsqueeze(-1)).squeeze(-1)  # (G, max_comp_len)
    token_lps = token_lps * comp_mask.float()

    counts = comp_mask.float().sum(-1).clamp(min=1)  # (G,)
    return token_lps.sum(-1) / counts  # (G,)


# ── Sampling ──────────────────────────────────────────────────────────────────


@torch.no_grad()
def sample_completions(
    model,
    tokenizer,
    example: dict,
    G: int,
    max_new_tokens: int,
    temperature: float,
) -> tuple[torch.Tensor, list[torch.Tensor], list[str]]:
    """
    Sample G completions for a single prompt.
    Returns:
        prompt_ids: (1, prompt_len) tensor (chat-templated, on model device)
        comp_ids_list: list of G tensors, each (1, comp_len_i)
        completions: list of G decoded strings
    """
    prompt_ids = tokenizer.apply_chat_template(
        make_messages(example["question"]),
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )["input_ids"].to(model.device)  # (1, prompt_len)

    prompt_repeated = prompt_ids.repeat_interleave(G, dim=0)  # (G, prompt_len)
    attn_mask = torch.ones_like(prompt_repeated)

    out = model.generate(
        input_ids=prompt_repeated,
        attention_mask=attn_mask,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )  # (G, prompt_len + comp_len_i)

    prompt_len = prompt_ids.shape[1]
    comp_ids_list = []
    completions = []
    for i in range(G):
        cids = out[i, prompt_len:].unsqueeze(0)  # (1, comp_len_i)
        comp_ids_list.append(cids)
        completions.append(tokenizer.decode(cids[0], skip_special_tokens=True))

    return prompt_ids, comp_ids_list, completions


# ── Training loop ─────────────────────────────────────────────────────────────


def grpo_train_step(
    policy,
    ref_model,
    tokenizer,
    optimizer,
    examples: list[dict],
    reward_fns: list[Callable[[dict, str], float]],
    config: GRPOConfig,
) -> dict:
    """One GRPO update step over a batch of prompts."""
    policy.train()
    ref_model.eval()

    total_loss = torch.tensor(0.0, device=DEVICE)
    all_metrics = []

    for example in examples:
        # 1. Sample G completions (returns token ids + decoded strings)
        prompt_ids, comp_ids_list, completions = sample_completions(
            policy, tokenizer, example, config.G, config.max_new_tokens, config.temperature,
        )

        # 2. Score with each reward function
        per_fn_scores: list[list[float]] = []
        for fn in reward_fns:
            per_fn_scores.append([fn(example, c) for c in completions])

        rewards = torch.tensor(
            [sum(per_fn_scores[j][i] for j in range(len(reward_fns))) for i in range(config.G)],
            dtype=torch.float32,
            device=DEVICE,
        )

        # 3. Batched log-prob computation: 1 forward pass each instead of G
        # Filter out empty completions
        non_empty = [i for i, cids in enumerate(comp_ids_list) if cids.shape[1] > 0]
        empty = [i for i in range(config.G) if i not in non_empty]

        policy_logps_buf = torch.zeros(config.G, device=DEVICE)
        ref_logps_buf = torch.zeros(config.G, device=DEVICE)

        if non_empty:
            ne_comp_ids = [comp_ids_list[i] for i in non_empty]
            policy_logps_ne = batch_completion_logprobs(policy, prompt_ids, ne_comp_ids, with_grad=True)
            ref_logps_ne = batch_completion_logprobs(ref_model, prompt_ids, ne_comp_ids, with_grad=False)
            for out_i, src_i in enumerate(non_empty):
                policy_logps_buf[src_i] = policy_logps_ne[out_i]
                ref_logps_buf[src_i] = ref_logps_ne[out_i]

        policy_logps = policy_logps_buf  # (G,)
        ref_logps = ref_logps_buf        # (G,)

        # 4. GRPO loss
        loss, metrics = compute_grpo_loss(
            policy_logps, ref_logps, rewards, beta=config.beta, epsilon=config.epsilon
        )

        # 5. Per-reward-function metrics
        for j, fn in enumerate(reward_fns):
            name = getattr(fn, "__name__", f"reward_fn_{j}")
            fn_scores = per_fn_scores[j]
            metrics[f"{name}/mean"] = sum(fn_scores) / len(fn_scores)
            metrics[f"{name}/accuracy"] = sum(s > 0 for s in fn_scores) / len(fn_scores)

        total_loss = total_loss + loss
        all_metrics.append(metrics)

    # 6. Backprop
    optimizer.zero_grad()
    (total_loss / len(examples)).backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    optimizer.step()

    def _avg(key):
        return sum(m[key] for m in all_metrics) / len(all_metrics)

    result = {
        "loss": total_loss.item() / len(examples),
        "mean_reward": _avg("mean_reward"),
        "mean_kl": _avg("kl"),
    }
    fn_keys = [k for k in all_metrics[0] if "/" in k]
    for k in fn_keys:
        result[k] = _avg(k)

    return result


# ── Benchmark ─────────────────────────────────────────────────────────────────
@torch.inference_mode()
def benchmark_throughput(
    model_name: str, tokenizer, n_new_tokens: int = 128, runs: int = 5
):
    """
    Compare tokens/sec and peak memory for bfloat16 vs 4-bit (NF4) generation.
    """
    import gc
    import time

    from transformers import AutoModelForCausalLM, BitsAndBytesConfig

    prompt_ids = tokenizer("What is the capital of France?", return_tensors="pt")

    def _run(model):
        device = next(model.parameters()).device
        ids = prompt_ids["input_ids"].to(device)
        with torch.inference_mode():  # warm-up
            model.generate(ids, max_new_tokens=16, do_sample=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        total_tokens = 0
        with torch.inference_mode():
            for _ in range(runs):
                out = model.generate(ids, max_new_tokens=n_new_tokens, do_sample=False)
                total_tokens += out.shape[1] - ids.shape[1]
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return total_tokens / (time.perf_counter() - t0)

    results = {}
    configs = [
        ("bfloat16", {"torch_dtype": torch.bfloat16, "device_map": "auto"}),
        (
            "4-bit NF4",
            {
                "quantization_config": BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type="nf4",
                ),
                "device_map": "auto",
            },
        ),
    ]
    for label, kwargs in configs:
        model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs).eval()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        tps = _run(model)
        mem_gb = (
            torch.cuda.max_memory_allocated() / 1e9
            if torch.cuda.is_available()
            else sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9
        )
        results[label] = {"tokens_per_sec": tps, "mem_gb": mem_gb}
        print(f"  {label:12s}  {tps:7.1f} tok/s   {mem_gb:.2f} GB")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    speedup = results["4-bit NF4"]["tokens_per_sec"] / results["bfloat16"]["tokens_per_sec"]
    mem_savings = 1 - results["4-bit NF4"]["mem_gb"] / results["bfloat16"]["mem_gb"]
    print(f"\n  4-bit speedup: {speedup:.2f}x  |  memory saved: {mem_savings:.0%}")

    return (
        "4-bit NF4"
        if results["4-bit NF4"]["tokens_per_sec"] > results["bfloat16"]["tokens_per_sec"]
        else "bfloat16"
    )


## Train

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "ibm-granite/granite-4.0-350m"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from unsloth import FastLanguageModel

policy = FastLanguageModel.from_pretrained(model_name, device=DEVICE)

In [ ]:
print("── Throughput benchmark ──────────────────────────────────────────────")
# result = benchmark_throughput(model_name, tokenizer)
result = "bfloat16"
print()

In [ ]:
if result == "4-bit NF4":
    print("Using 4-bit NF4 quantization for training based on benchmark results.")
    from transformers import BitsAndBytesConfig

    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,  # nested quantization saves a bit more
        bnb_4bit_quant_type="nf4",  # NF4 is optimal for normally-distributed weights
    )
else:
    print("Using bfloat16 for training based on benchmark results.")
    bnb_cfg = None

policy = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_cfg,
    device_map="auto",
)
ref_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_cfg,
    device_map="auto",
).eval()

if result == "bfloat16":
    print("Casting models to bfloat16 for training.")
    policy = policy.to(dtype=torch.bfloat16)
    ref_model = ref_model.to(dtype=torch.bfloat16)

# Gradient checkpointing: trade compute for memory, enabling larger G or batch size
policy.gradient_checkpointing_enable()

for p in ref_model.parameters():
    p.requires_grad_(False)


In [ ]:
import re

from datasets import DatasetDict, load_dataset


def extract_gold(example: dict) -> str:
    """Extract the gold answer from a GSM8K example."""
    answer = example["answer"]
    match = re.search(r"####\s*(-?\d+(?:,\d{3})*(?:\.\d+)?)\s*$", answer)
    if not match:
        raise ValueError(f"Could not extract gold answer from: {answer!r}")
    return match.group(1).replace(",", "")


ds = cast(DatasetDict, load_dataset("openai/gsm8k", "main"))

ds["train"] = ds["train"].map(lambda ex: {"gold": extract_gold(ex)})
ds["test"] = ds["test"].map(lambda ex: {"gold": extract_gold(ex)})

In [ ]:
def reward_correctness(example: dict, completion: str) -> float:
    return 5.0 if example["gold"] in completion else 0.0


def reward_brevity(example: dict, completion: str) -> float:
    # target between 100 and 300 chars, with a penalty outside that range
    length = len(completion)
    if length < 100:
        return -0.01 * (100 - length) + 1
    elif length > 300:
        return -0.012 * (length - 300) + 1
    else:
        return 1


reward_fns = [reward_correctness, reward_brevity]

In [ ]:
# with torch.inference_mode():
#     messages = [
#         {
#             "role": "user",
#             "content": "There are 87 oranges and 290 bananas in Philip's collection. If the bananas are organized into 2 groups and oranges are organized into 93 groups",
#         }
#     ]

#     inputs = tokenizer.apply_chat_template(
#         messages,
#         add_generation_prompt=True,
#         tokenize=True,
#         return_dict=True,
#         return_tensors="pt",
#     ).to(policy.device)

#     out = policy.generate(
#         **inputs,
#         max_new_tokens=512,
#         temperature=0.9,
#         do_sample=True,
#         pad_token_id=tokenizer.eos_token_id,
#     )

#     print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
optimizer = AdamW(policy.parameters(), lr=1e-4)
config = GRPOConfig(G=2, beta=0.01, max_new_tokens=256, temperature=0.9)


In [ ]:
from tqdm import tqdm


def batchify(dataset, batch_size):
    batch = []
    for example in dataset:
        batch.append(example)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


BATCH_SIZE = 2

for epoch in range(1):
    pbar = tqdm(batchify(ds["train"], batch_size=BATCH_SIZE), total=len(ds["train"]) // BATCH_SIZE)
    for batch in pbar:
        batch = cast(list[dict], batch)
        metrics = grpo_train_step(
            policy, ref_model, tokenizer, optimizer, batch, reward_fns, config
        )
        fn_stats = {k: f"{v:.3f}" for k, v in metrics.items() if "/" in k}
        print(
            {
                "loss": f"{metrics['loss']:.4f}",
                "reward": f"{metrics['mean_reward']:.3f}",
                "kl": f"{metrics['mean_kl']:.4f}",
                **fn_stats,
            }
        )